# StructuredAlpha — Results Dashboard

Run `run_all.py` first to generate the data files this notebook reads.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

RESULTS = os.path.join('..', 'results')
DATA    = os.path.join('..', 'data')

## 1 — Loan Pool

In [ ]:
from loan_pool import generate_loan_pool

loans = generate_loan_pool()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(loans['original_principal'] / 1e6, bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Loan Principal Distribution')
axes[0].set_xlabel('Principal ($M)')
axes[0].set_ylabel('Count')

rating_counts = loans['rating'].value_counts()
axes[1].pie(rating_counts, labels=rating_counts.index, autopct='%1.0f%%',
            colors=['#4878CF', '#6ACC65', '#D65F5F'])
axes[1].set_title('Rating Breakdown')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'plot1_loan_pool.png'), bbox_inches='tight')
plt.show()
print(f'Total notional: ${loans["original_principal"].sum()/1e6:.0f}M  |  Loans: {len(loans)}')

## 2 — Single-Path Waterfall Cashflows

In [ ]:
from waterfall import fresh_tranches, apply_waterfall, TRANCHE_DEFS
from monte_carlo import run_single_path

rng  = np.random.default_rng(42)
path = run_single_path(cdr=0.025, recovery_rate=0.50, cpr=0.15, rng=rng)

periods = range(1, len(path['equity_flows']) + 1)

fig, ax = plt.subplots(figsize=(13, 5))
colors  = ['#2166ac', '#4dac26', '#d01c8b', '#f1a340', '#fee090']
labels  = ['AAA interest', 'AA interest', 'A interest', 'BBB interest', 'Equity']

from config import SOFR
stacks = []
for d in TRANCHE_DEFS:
    monthly_int = d['initial_balance'] * (SOFR + d['spread']) / 12
    stacks.append([monthly_int] * 60)
stacks.append(path['equity_flows'])

bottom = np.zeros(60)
for stack, label, color in zip(stacks, labels, colors):
    ax.bar(periods, stack, bottom=bottom, label=label, color=color, width=0.8)
    bottom += np.array(stack)

ax.set_title('Waterfall Cash Distribution — Base Case (CDR=2.5%, Recovery=50%)')
ax.set_xlabel('Period (months)')
ax.set_ylabel('Cash ($)')
ax.legend(loc='upper right')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'plot2_waterfall.png'), bbox_inches='tight')
plt.show()

## 3 — Monte Carlo OC Ratio Distribution

In [ ]:
mc = pd.read_csv(os.path.join(RESULTS, 'mc_summary.csv'))
oc_series = np.load(os.path.join(RESULTS, 'mc_oc_series.npy'))  # (2000, 60)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Min OC per path
axes[0].hist(mc['min_bbb_oc'], bins=40, color='#4878CF', edgecolor='white')
axes[0].axvline(1.03, color='red', linestyle='--', linewidth=1.5, label='OC threshold 1.03x')
axes[0].set_title('Distribution of Minimum BBB OC Ratio (2,000 paths)')
axes[0].set_xlabel('Minimum BBB OC Ratio')
axes[0].set_ylabel('Count')
axes[0].legend()

# Breach rate vs CDR
cdr_bins = pd.cut(mc['cdr'], bins=np.arange(0, 0.11, 0.01))
breach_by_cdr = mc.groupby(cdr_bins, observed=False)['oc_breached_bbb'].mean()
axes[1].bar(range(len(breach_by_cdr)), breach_by_cdr.values, color='#D65F5F', edgecolor='white')
axes[1].axhline(0.20, color='black', linestyle='--', linewidth=1, label='20% threshold')
axes[1].set_xticks(range(len(breach_by_cdr)))
axes[1].set_xticklabels([f'{b.right:.0%}' for b in breach_by_cdr.index], rotation=45)
axes[1].set_title('BBB OC Breach Rate by CDR Band')
axes[1].set_xlabel('CDR (upper bound of bin)')
axes[1].set_ylabel('Breach Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'plot3_mc_oc.png'), bbox_inches='tight')
plt.show()

stress_mask = mc['cdr'] > 0.08
print(f"Overall BBB breach rate : {mc['oc_breached_bbb'].mean():.1%}")
print(f"BBB breach rate CDR>8%  : {mc.loc[stress_mask,'oc_breached_bbb'].mean():.1%}")

## 4 — LSTM Training Curves

In [ ]:
history = pd.read_csv(os.path.join(RESULTS, 'lstm_history.csv'))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train_loss'], label='Train loss', color='#4878CF')
ax.plot(history['val_loss'],   label='Val loss',   color='#D65F5F')
ax.set_title('LSTM Regime Classifier — Training Curves')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'plot4_lstm_training.png'), bbox_inches='tight')
plt.show()

## 5 — Regime Classification Results

In [ ]:
preds_df     = pd.read_csv(os.path.join(RESULTS, 'lstm_predictions.csv'))
regime_names = ['TIGHT', 'NORMAL', 'WIDE']

cm  = confusion_matrix(preds_df['actual_idx'], preds_df['predicted_idx'])
acc = (preds_df['actual_idx'] == preds_df['predicted_idx']).mean()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=regime_names, yticklabels=regime_names, ax=ax)
ax.set_title(f'Confusion Matrix — Test Set Accuracy: {acc:.1%}')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'plot5_confusion.png'), bbox_inches='tight')
plt.show()
print(f'Test accuracy: {acc:.1%}')

## 6 — RL Training Reward Curve

In [ ]:
log_df = pd.read_csv(os.path.join(RESULTS, 'rl_training_log.csv'))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(log_df.index + 1, log_df['episode_reward'], color='#4878CF', alpha=0.3, linewidth=0.5)
window = 200
ax.plot(log_df.index + 1,
        log_df['episode_reward'].rolling(window, min_periods=1).mean(),
        color='#D65F5F', linewidth=2, label=f'{window}-ep rolling mean')
ax.set_title('PPO Training — Episode Reward (6,826 game episodes)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'plot6_rl_training.png'), bbox_inches='tight')
plt.show()

## 7 — Baseline vs RL Policy Comparison

In [ ]:
baseline_df = pd.read_csv(os.path.join(RESULTS, 'rl_baseline.csv'))
policy_df   = pd.read_csv(os.path.join(RESULTS, 'rl_policy.csv'))
summary_txt = open(os.path.join(RESULTS, 'rl_summary.txt')).read()
print(summary_txt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Return distributions
axes[0].hist(baseline_df['episode_return'], bins=40, alpha=0.6,
             color='#4878CF', label='Equal-weight baseline')
axes[0].hist(policy_df['episode_return'],   bins=40, alpha=0.6,
             color='#D65F5F', label='PPO policy')
axes[0].axvline(baseline_df['episode_return'].mean(), color='#4878CF',
                linestyle='--', linewidth=1.5)
axes[0].axvline(policy_df['episode_return'].mean(),   color='#D65F5F',
                linestyle='--', linewidth=1.5)
axes[0].set_title('Episode Return Distribution (500 evaluation episodes)')
axes[0].set_xlabel('Episode Total Return')
axes[0].set_ylabel('Count')
axes[0].legend()

# Sharpe: divide episode total by 60 to get avg monthly return, then annualise
def _sharpe(ep_returns):
    monthly = np.array(ep_returns) / 60
    return monthly.mean() / monthly.std() * np.sqrt(12) if monthly.std() > 0 else 0.0

sharpes = [_sharpe(baseline_df['episode_return']), _sharpe(policy_df['episode_return'])]
bars = axes[1].bar(['Equal-weight\n(baseline)', 'PPO policy'],
                    sharpes, color=['#4878CF', '#D65F5F'], width=0.5)
for bar, val in zip(bars, sharpes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                 f'{val:.2f}', ha='center', fontweight='bold')
axes[1].set_title('Sharpe Ratio Comparison (annualised)')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].set_ylim(0, max(sharpes) * 1.25)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'plot7_strategy_comparison.png'), bbox_inches='tight')
plt.show()